# NHTSA Recall Lag Analysis: Data Cleaning Pipeline

This notebook takes the raw NHTSA complaint and recall flat files, cleans them step by step, and loads
the result into SQL Server (database `NHTSA_v2`, schema `v2`) for the SQL analysis.

**Steps**

1. Configure paths and the SQL Server target.
2. Define the column layout of the raw files.
3. Load only the columns the analysis needs.
4. Keep vehicle records only (`PROD_TYPE = 'V'`, `RCLTYPECD = 'V'`), dropping tires, child seats and equipment.
5. Parse dates and clean numeric and flag fields.
6. Map each vehicle make to its parent manufacturer (`LEXUS` → TOYOTA, `JEEP` → CHRYSLER).
7. Normalise model names so `F-150` and `F150` match.
8. Reduce component codes to a shared top-level vocabulary.
9. Collapse duplicate lines so each complaint counts once per vehicle and component.
10. Build the final tables, validate them, and load them into SQL Server.

Every filtering step logs rows in and out, so the cleaning is fully auditable.

## 0. Configuration

In [1]:
from pathlib import Path
import csv
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)


def find_project_root(start: Path) -> Path:
    """Walk up from the notebook folder until the folder holding `datasets/` is found."""
    for p in [start, *start.parents]:
        if (p / "datasets").is_dir():
            return p
    raise FileNotFoundError(f"No 'datasets' folder found above {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "datasets"
OUT_DIR = PROJECT_ROOT / "Results" / "cleaning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# SQL Server target database and schema.
# The database files go on D: because C: is nearly full; the folder is created if missing.
SQL_SERVER = "localhost"
SQL_DATABASE = "NHTSA_v2"
SQL_DATA_DIR = r"D:\SQLData"
SQL_SCHEMA = "v2"
LOAD_TO_SQL = True
# Complaint narratives are ~1.5 GB and not used by this analysis. Set True to load them
# into v2.complaint_text for future text analysis (adds several minutes to the load).
LOAD_COMPLAINT_TEXT = False

# Lineage log: every filtering step records rows in and out, saved at the end.
lineage = []
def log_step(table, step, rows_before, rows_after, note=""):
    lineage.append({"table": table, "step": step, "rows_before": rows_before,
                    "rows_after": rows_after, "rows_removed": rows_before - rows_after, "note": note})
    print(f"[{table}] {step}: {rows_before:,} -> {rows_after:,} ({rows_before - rows_after:,} removed) {note}")

print("Project root:", PROJECT_ROOT)

Project root: D:\DA project\NHTSA PROJECT


## 1. Source schemas

NHTSA ships both files as tab-delimited text with no header row. Column names come from the NHTSA
field dictionaries (`CMPL.txt`, `RCL.txt`).

In [2]:
CMPL_COLUMNS = [
    "CMPLID", "ODINO", "MFR_NAME", "MAKETXT", "MODELTXT", "YEARTXT", "CRASH", "FAILDATE", "FIRE",
    "INJURED", "DEATHS", "COMPDESC", "CITY", "STATE", "VIN", "DATEA", "LDATE", "MILES", "OCCURENCES",
    "CDESCR", "CMPL_TYPE", "POLICE_RPT_YN", "PURCH_DT", "ORIG_OWNER_YN", "ANTI_BRAKES_YN",
    "CRUISE_CONT_YN", "NUM_CYLS", "DRIVE_TRAIN", "FUEL_SYS", "FUEL_TYPE", "TRANS_TYPE", "VEH_SPEED",
    "DOT", "TIRE_SIZE", "LOC_OF_TIRE", "TIRE_FAIL_TYPE", "ORIG_EQUIP_YN", "MANUF_DT", "SEAT_TYPE",
    "RESTRAINT_TYPE", "DEALER_NAME", "DEALER_TEL", "DEALER_CITY", "DEALER_STATE", "DEALER_ZIP",
    "PROD_TYPE", "REPAIRED_YN", "MEDICAL_ATTN", "VEHICLES_TOWED_YN", "STATE_OF_INCIDENT",
    "VEHICLE_OPERATOR",
]

RCL_COLUMNS = [
    "RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "MFGCAMPNO", "COMPNAME", "MFGNAME",
    "BGMAN", "ENDMAN", "RCLTYPECD", "POTAFF", "ODATE", "INFLUENCED_BY", "MFGTXT", "RCDATE", "DATEA",
    "RPNO", "FMVSS", "DESC_DEFECT", "CONEQUENCE_DEFECT", "CORRECTIVE_ACTION", "NOTES", "RCL_CMPT_ID",
    "MFR_COMP_NAME", "MFR_COMP_DESC", "MFR_COMP_PTNO", "DO_NOT_DRIVE", "PARK_OUTSIDE",
]
assert len(CMPL_COLUMNS) == 51 and len(RCL_COLUMNS) == 29

## 2. Load raw files

Only the columns the analysis needs are read, as text, so nothing is silently converted
(e.g. model year `1987` becoming `1987.0`). Text is stored as Arrow strings, which take a fraction
of the memory of Python objects; that matters on an 8 GB machine with SQL Server running alongside.

In [3]:
read_opts = dict(sep="\t", header=None, quoting=csv.QUOTE_NONE, dtype="string[pyarrow]",
                 encoding="utf-8", encoding_errors="replace", keep_default_na=False, na_values=[""])

cmpl_usecols = ["CMPLID", "ODINO", "MFR_NAME", "MAKETXT", "MODELTXT", "YEARTXT", "CRASH", "FAILDATE",
                "FIRE", "INJURED", "DEATHS", "COMPDESC", "DATEA", "PROD_TYPE"]
if LOAD_COMPLAINT_TEXT:
    cmpl_usecols.append("CDESCR")

t0 = time.time()
complaints_raw = pd.read_csv(DATA_DIR / "FLAT_CMPL.txt", names=CMPL_COLUMNS, usecols=cmpl_usecols, **read_opts)
print(f"Complaints: {complaints_raw.shape} in {time.time() - t0:.0f}s")

rcl_usecols = ["RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "COMPNAME", "MFGNAME", "RCLTYPECD",
               "POTAFF", "INFLUENCED_BY", "RCDATE", "DESC_DEFECT", "CONEQUENCE_DEFECT",
               "DO_NOT_DRIVE", "PARK_OUTSIDE"]
# The pre/post-2010 split is only how NHTSA stores the history; the schema is identical.
recalls_raw = pd.concat(
    [pd.read_csv(DATA_DIR / f, names=RCL_COLUMNS, usecols=rcl_usecols, **read_opts)
     for f in ["FLAT_RCL_PRE_2010.txt", "FLAT_RCL_POST_2010.txt"]],
    ignore_index=True,
)
print(f"Recalls: {recalls_raw.shape}, {recalls_raw['CAMPNO'].nunique():,} campaigns")

# Strip stray whitespace on every text field once, up front.
for df in (complaints_raw, recalls_raw):
    for col in df.columns:
        df[col] = df[col].str.strip()


def is_value(s: pd.Series, value: str) -> pd.Series:
    """s == value, with missing treated as False (Arrow string comparisons return NA for missing)."""
    return s.eq(value).fillna(False).astype(bool)

Complaints: (2234970, 14) in 35s
Recalls: (326100, 15), 30,238 campaigns


## 3. Keep vehicle records only

Both sources mix vehicles with tires, child seats and aftermarket equipment. The analysis is about vehicle
recalls, so only vehicle records are kept on both sides.

In [4]:
print(complaints_raw["PROD_TYPE"].value_counts(dropna=False).to_string(), "\n")
print(recalls_raw.drop_duplicates("CAMPNO")["RCLTYPECD"].value_counts(dropna=False).to_string())

n = len(complaints_raw)
complaints = complaints_raw[is_value(complaints_raw["PROD_TYPE"], "V")].copy()
log_step("complaints", "vehicle records only (PROD_TYPE = V)", n, len(complaints))

n = len(recalls_raw)
recalls = recalls_raw[is_value(recalls_raw["RCLTYPECD"], "V")].copy()
log_step("recalls", "vehicle recalls only (RCLTYPECD = V)", n, len(recalls))

del complaints_raw, recalls_raw

PROD_TYPE
V       2162988
T         41434
C         15272
E         15257
<NA>         19 

RCLTYPECD
V    26276
E     2859
T      787
C      252
I       53
X       11
[complaints] vehicle records only (PROD_TYPE = V): 2,234,970 -> 2,162,988 (71,982 removed) 
[recalls] vehicle recalls only (RCLTYPECD = V): 326,100 -> 285,005 (41,095 removed) 


## 4. Dates

- `DATEA` (complaint received by NHTSA) is the complaint date: complete coverage, set by NHTSA.
- `RCDATE` is the recall date.
- `FAILDATE` (when the owner says the failure happened) is self-reported; it's kept as an optional field,
  with the `19010101` placeholder and impossible values set to missing.

In [5]:
def parse_yyyymmdd(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.str[:8], format="%Y%m%d", errors="coerce")

complaints["COMPLAINT_DATE"] = parse_yyyymmdd(complaints["DATEA"])
complaints["FAIL_DATE"] = parse_yyyymmdd(complaints["FAILDATE"])
recalls["RECALL_DATE"] = parse_yyyymmdd(recalls["RCDATE"])

EXTRACT_DATE = max(complaints["COMPLAINT_DATE"].max(), recalls["RECALL_DATE"].max())
TODAY = pd.Timestamp.today().normalize()
print("Complaint dates:", complaints["COMPLAINT_DATE"].min().date(), "to", complaints["COMPLAINT_DATE"].max().date())
print("Recall dates:   ", recalls["RECALL_DATE"].min().date(), "to", recalls["RECALL_DATE"].max().date())
print("Data extract date:", EXTRACT_DATE.date())

n = len(complaints)
complaints = complaints[complaints["COMPLAINT_DATE"].notna() & (complaints["COMPLAINT_DATE"] <= TODAY)]
log_step("complaints", "valid, non-future complaint date", n, len(complaints))
n = len(recalls)
recalls = recalls[recalls["RECALL_DATE"].notna() & (recalls["RECALL_DATE"] <= TODAY)]
log_step("recalls", "valid, non-future recall date", n, len(recalls))

# FAIL_DATE: drop placeholder (1901) and values after the complaint was filed.
bad_fail = (complaints["FAIL_DATE"] < "1950-01-01") | (complaints["FAIL_DATE"] > complaints["COMPLAINT_DATE"])
print(f"FAIL_DATE set to missing: {bad_fail.sum():,} placeholder/impossible values")
complaints.loc[bad_fail, "FAIL_DATE"] = pd.NaT

Complaint dates: 1995-01-03 to 2026-08-12
Recall dates:    1966-01-19 to 2026-08-10
Data extract date: 2026-08-12
[complaints] valid, non-future complaint date: 2,162,988 -> 2,162,988 (0 removed) 
[recalls] valid, non-future recall date: 285,005 -> 285,005 (0 removed) 
FAIL_DATE set to missing: 6,391 placeholder/impossible values


## 5. Numeric and flag fields

In [6]:
def to_model_year(s: pd.Series) -> pd.Series:
    year = pd.to_numeric(s, errors="coerce").astype("Int64")
    # 9999 is NHTSA's "unknown"; anything outside the plausible range is treated the same way.
    return year.where((year >= 1949) & (year <= EXTRACT_DATE.year + 2))

complaints["MODEL_YEAR"] = to_model_year(complaints["YEARTXT"])
recalls["MODEL_YEAR"] = to_model_year(recalls["YEARTXT"])

complaints["CRASH"] = is_value(complaints["CRASH"], "Y").astype("int8")
complaints["FIRE"] = is_value(complaints["FIRE"], "Y").astype("int8")
for col in ["INJURED", "DEATHS"]:
    complaints[col] = pd.to_numeric(complaints[col], errors="coerce").fillna(0).astype("int32")

recalls["UNITS_AFFECTED"] = pd.to_numeric(recalls["POTAFF"], errors="coerce").astype("Int64")
recalls["DO_NOT_DRIVE"] = is_value(recalls["DO_NOT_DRIVE"], "Yes").astype("int8")
recalls["PARK_OUTSIDE"] = is_value(recalls["PARK_OUTSIDE"], "Yes").astype("int8")

print("Complaint model year missing:", f"{complaints['MODEL_YEAR'].isna().mean():.2%}")
print("Recall model year missing:   ", f"{recalls['MODEL_YEAR'].isna().mean():.2%}")
print("INFLUENCED_BY:", recalls.drop_duplicates("CAMPNO")["INFLUENCED_BY"].value_counts(dropna=False).to_dict())

Complaint model year missing: 0.69%
Recall model year missing:    0.20%
INFLUENCED_BY: {'MFR': 22013, 'ODI': 3584, 'OVSC': 677, 'ISSUE_INVGSTN': 2}


## 6. Manufacturer: vehicle make → parent OEM

The legal-entity name field (`MFR_NAME` / `MFGNAME`) has thousands of spellings and includes merged
companies (e.g. `DAIMLERCHRYSLER` covers both Chrysler and Mercedes vehicles), so it is not reliable for
grouping. The make field (`CHEVROLET`, `LEXUS`, `JEEP`) is short and clean, so each make is mapped to
its parent manufacturer with an explicit lookup table. `CHRYSLER` is used as the parent label for
Dodge, Jeep, Ram and Fiat.

What this step does:
- builds the make → OEM lookup,
- removes Volvo heavy trucks, which share the `VOLVO` make,
- saves the makes that did not match to `Results/cleaning/unmapped_makes.csv` for review,
- keeps only rows that map to a passenger-vehicle manufacturer.

Makes not in the lookup are heavy trucks, buses, RVs, trailers and motorcycles, which are out of scope.

In [7]:
OEM_BRANDS = {
    "GENERAL MOTORS": ["CHEVROLET", "GMC", "CADILLAC", "BUICK", "PONTIAC", "OLDSMOBILE", "SATURN", "HUMMER", "GEO"],
    "FORD": ["FORD", "LINCOLN", "MERCURY"],
    "CHRYSLER": ["CHRYSLER", "DODGE", "JEEP", "RAM", "PLYMOUTH", "EAGLE", "FIAT", "ALFA ROMEO", "ALFA", "SRT"],
    "TOYOTA": ["TOYOTA", "LEXUS", "SCION"],
    "HONDA": ["HONDA", "ACURA"],
    "NISSAN": ["NISSAN", "INFINITI", "DATSUN"],
    "HYUNDAI": ["HYUNDAI", "GENESIS"],
    "KIA": ["KIA"],
    "VOLKSWAGEN": ["VOLKSWAGEN", "AUDI", "BENTLEY", "LAMBORGHINI"],
    "BMW": ["BMW", "MINI", "ROLLS-ROYCE"],
    "MERCEDES-BENZ": ["MERCEDES-BENZ", "MERCEDES BENZ", "MERCEDES", "MERCEDES-MAYBACH", "MAYBACH", "SMART"],
    "SUBARU": ["SUBARU"],
    "MAZDA": ["MAZDA"],
    "MITSUBISHI": ["MITSUBISHI"],
    "TESLA": ["TESLA"],
    "VOLVO": ["VOLVO"],
    "POLESTAR": ["POLESTAR"],
    "JAGUAR LAND ROVER": ["JAGUAR", "LAND ROVER"],
    "PORSCHE": ["PORSCHE"],
    "SUZUKI": ["SUZUKI"],
    "SAAB": ["SAAB"],
    "ISUZU": ["ISUZU"],
    "RIVIAN": ["RIVIAN"],
    "LUCID": ["LUCID"],
    "MASERATI": ["MASERATI"],
    "FERRARI": ["FERRARI"],
    "ASTON MARTIN": ["ASTON MARTIN"],
    "MCLAREN": ["MCLAREN"],
    "LOTUS": ["LOTUS"],
    "FISKER": ["FISKER", "KARMA"],
    "VINFAST": ["VINFAST"],
    "DAEWOO": ["DAEWOO", "DAEWOO PUERTO RICO"],
    "AMERICAN MOTORS": ["AMERICAN MOTORS", "RENAULT"],   # AMC sold Renault models in the US in the 1980s
    "PEUGEOT": ["PEUGEOT"],
}
# The audit printed below (and saved to Results/cleaning/unmapped_makes.csv) is how these
# small brands were found; what's left unmapped is motorcycles, RVs, trucks and buses.

def make_key(s: pd.Series) -> pd.Series:
    """Uppercase, letters and digits only: 'Mercedes Benz' and 'MERCEDES-BENZ' give the same key."""
    return s.fillna("").str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

MAKE_TO_OEM = {k: oem for oem, brands in OEM_BRANDS.items()
               for k in make_key(pd.Series(brands))}

# Volvo heavy trucks share the VOLVO make (VNL, VNM, VHD, model 770...). Drop them by model pattern.
VOLVO_TRUCK_MODEL = r"^(VN|VH|VT|VAH|WG|WI|WX|WC|AC|ACL|7[0-9]0$)"

for name, df in [("complaints", complaints), ("recalls", recalls)]:
    df["VEHICLE_MAKE"] = df["MAKETXT"].str.upper()
    df["MFR_NAME_CLEAN"] = make_key(df["MAKETXT"]).map(MAKE_TO_OEM)
    volvo_truck = (is_value(df["MFR_NAME_CLEAN"].astype("string"), "VOLVO")
                   & make_key(df["MODELTXT"]).str.match(VOLVO_TRUCK_MODEL).fillna(False).astype(bool))
    df.loc[volvo_truck, "MFR_NAME_CLEAN"] = np.nan
    print(f"{name}: Volvo heavy-truck rows excluded: {volvo_truck.sum():,}")

# Audit what the lookup leaves out, largest first, before filtering.
unmapped_c = complaints.loc[complaints["MFR_NAME_CLEAN"].isna(), "VEHICLE_MAKE"].value_counts()
unmapped_r = recalls.loc[recalls["MFR_NAME_CLEAN"].isna()].drop_duplicates("CAMPNO")["VEHICLE_MAKE"].value_counts()
print("\nLargest unmapped complaint makes:\n", unmapped_c.head(15).to_string())
print("\nLargest unmapped recall makes (campaigns):\n", unmapped_r.head(15).to_string())
pd.concat({"complaint_rows": unmapped_c, "recall_campaigns": unmapped_r}, axis=1).fillna(0).astype(int) \
  .to_csv(OUT_DIR / "unmapped_makes.csv")

n = len(complaints)
complaints = complaints[complaints["MFR_NAME_CLEAN"].notna()].copy()
log_step("complaints", "make maps to a passenger-vehicle OEM", n, len(complaints))
n = len(recalls)
recalls = recalls[recalls["MFR_NAME_CLEAN"].notna()].copy()
log_step("recalls", "make maps to a passenger-vehicle OEM", n, len(recalls))

print("\nComplaint rows by OEM:\n", complaints["MFR_NAME_CLEAN"].value_counts().head(12).to_string())

complaints: Volvo heavy-truck rows excluded: 2,023
recalls: Volvo heavy-truck rows excluded: 1,369

Largest unmapped complaint makes:
 VEHICLE_MAKE
UNKNOWN             10003
HARLEY-DAVIDSON      4152
YAMAHA               2573
FLEETWOOD            2428
FOREST RIVER         2344
VOLVO                2023
WINNEBAGO            1766
FREIGHTLINER         1592
KAWASAKI             1420
KEYSTONE             1139
DUCATI               1006
GRAND DESIGN          950
CAN-AM                938
THOR MOTOR COACH      759
HOLIDAY RAMBLER       677

Largest unmapped recall makes (campaigns):
 VEHICLE_MAKE
FREIGHTLINER               538
MACK                       426
BLUE BIRD                  420
FOREST RIVER               403
INTERNATIONAL HARVESTER    265
KENWORTH                   242
INTERNATIONAL              240
WINNEBAGO                  239
FLEETWOOD                  234
THOMAS BUILT BUSES         227
PETERBILT                  226
VOLVO                      207
ALTEC                      202
K

## 7. Model name normalisation

Complaint and recall model names are typed differently (`F-150` vs `F150`, `MX-5 MIATA` vs `MX5 MIATA`).
`MODEL_NORM` keeps letters and digits only. The raw name is kept in `MODEL_NAME`.

In [8]:
for df in (complaints, recalls):
    df["MODEL_NAME"] = df["MODELTXT"].str.upper()
    df["MODEL_NORM"] = make_key(df["MODELTXT"])
    df.loc[df["MODEL_NORM"].isin(["", "UNKNOWN", "UNKNOWN1", "NA", "OTHER"]), "MODEL_NORM"] = np.nan

recall_keys = set(zip(make_key(recalls["VEHICLE_MAKE"]), recalls["MODEL_NORM"]))
has_model = complaints["MODEL_NORM"].notna()
in_recalls = pd.Series(list(zip(make_key(complaints["VEHICLE_MAKE"]), complaints["MODEL_NORM"])),
                       index=complaints.index).isin(recall_keys)
print(f"Complaint rows with a usable model name: {has_model.mean():.2%}")
print(f"...of those, make+model also appears in recall data: {in_recalls[has_model].mean():.2%}")

Complaint rows with a usable model name: 100.00%
...of those, make+model also appears in recall data: 98.65%


## 8. Component vocabulary

Both sources use a colon-delimited hierarchy (`AIR BAGS:FRONTAL:SENSOR`). The top level (text before the
first colon) is used as the shared vocabulary between complaints and recalls. True duplicates are merged,
and free-text junk from car-seat forms is measured and then dropped.

In [9]:
for df, col in [(complaints, "COMPDESC"), (recalls, "COMPNAME")]:
    df["COMPONENT_RAW"] = df[col].str.upper()
    df["COMPONENT_CLEAN"] = df["COMPONENT_RAW"].str.split(":").str[0].str.strip()

COMPONENT_ALIASES = {
    "ENGINE AND ENGINE COOLING": "ENGINE",
    "VISIBILITY/WIPER": "VISIBILITY",
    "COMMUNICATIONS": "COMMUNICATION",
    "ELECTRONIC STABILITY CONTROL": "ELECTRONIC STABILITY CONTROL (ESC)",
    "SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD": "SERVICE BRAKES, HYDRAULIC",
}
for df in (complaints, recalls):
    df["COMPONENT_CLEAN"] = df["COMPONENT_CLEAN"].replace(COMPONENT_ALIASES)

recall_vocab = set(recalls["COMPONENT_CLEAN"].dropna())
not_in_recalls = complaints.loc[~complaints["COMPONENT_CLEAN"].isin(recall_vocab), "COMPONENT_CLEAN"]
print("Complaint components with no recall equivalent:\n", not_in_recalls.value_counts(dropna=False).to_string())

# Keep legacy categories (FIRERELATED, ROLLOVER, OTHER) for severity stats; drop only free text / blanks.
LEGACY = {"FIRERELATED", "ROLLOVER", "OTHER", "UNKNOWN OR OTHER"}
junk = complaints["COMPONENT_CLEAN"].isna() | (
    ~complaints["COMPONENT_CLEAN"].isin(recall_vocab | LEGACY))
n = len(complaints)
complaints = complaints[~junk].copy()
log_step("complaints", "drop blank / free-text component", n, len(complaints))

Complaint components with no recall equivalent:
 COMPONENT_CLEAN
FIRERELATED                                      154
CHEST CLIP, BUCKLE, HARNESS                      148
<NA>                                             134
OTHER/I AM NOT SURE                               95
CARRY HANDLE, SHELL, BASE                         84
TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)     40
INSERT, PADDING                                   36
OTHER/UNKNOWN                                     10
NONE                                               5
I SUSPECT THE CAR SEAT IS COUNTERFEIT              4
OTHER                                              2
ROLLOVER                                           1
[complaints] drop blank / free-text component: 2,108,157 -> 2,107,601 (556 removed) 


## 9. Collapse duplicate lines

A complaint (`ODINO`) gets one row per component line it names, and several lines collapse to the same
top-level component (`AIR BAGS:FRONTAL` and `AIR BAGS:SIDE` both become `AIR BAGS`). Those duplicates
are removed so a complaint counts once per vehicle and component. Recalls are de-duplicated the same way.

In [10]:
cmpl_key = ["ODINO", "VEHICLE_MAKE", "MODEL_NORM", "MODEL_YEAR", "COMPONENT_CLEAN"]
n = len(complaints)
complaints = complaints.sort_values("CMPLID", key=lambda s: s.astype("int64")) \
                       .drop_duplicates(cmpl_key, keep="first")
log_step("complaints", "one row per complaint x vehicle x component", n, len(complaints))

rcl_key = ["CAMPNO", "VEHICLE_MAKE", "MODEL_NORM", "MODEL_YEAR", "COMPONENT_CLEAN"]
n = len(recalls)
recalls = recalls.drop_duplicates(rcl_key, keep="first")
log_step("recalls", "one row per campaign x vehicle x component", n, len(recalls))

print(f"\nRows per complaint after collapsing: {len(complaints) / complaints['ODINO'].nunique():.2f}")
print(f"Distinct complaints (ODINO): {complaints['ODINO'].nunique():,}")
print(f"Distinct recall campaigns:   {recalls['CAMPNO'].nunique():,}")

[complaints] one row per complaint x vehicle x component: 2,107,601 -> 2,042,264 (65,337 removed) 
[recalls] one row per campaign x vehicle x component: 149,322 -> 59,560 (89,762 removed) 

Rows per complaint after collapsing: 1.32
Distinct complaints (ODINO): 1,550,612
Distinct recall campaigns:   11,059


## 10. Final tables

Column names are final here; the SQL scripts use them without renaming.

- `complaints`: one row per complaint × vehicle × component
- `recalls`: one row per campaign × vehicle × component
- `recall_text`: one row per campaign (defect and consequence text, used by the air bag classification)
- `complaint_text` (optional): one row per complaint narrative

In [11]:
complaints_final = pd.DataFrame({
    "CMPLID": complaints["CMPLID"].astype("int64"),
    "ODINO": complaints["ODINO"].astype("int64"),
    "MFR_NAME_RAW": complaints["MFR_NAME"].str.upper(),
    "MFR_NAME_CLEAN": complaints["MFR_NAME_CLEAN"],
    "VEHICLE_MAKE": complaints["VEHICLE_MAKE"],
    "MODEL_NAME": complaints["MODEL_NAME"],
    "MODEL_NORM": complaints["MODEL_NORM"],
    "MODEL_YEAR": complaints["MODEL_YEAR"],
    "COMPONENT_RAW": complaints["COMPONENT_RAW"],
    "COMPONENT_CLEAN": complaints["COMPONENT_CLEAN"],
    "COMPLAINT_DATE": complaints["COMPLAINT_DATE"],
    "FAIL_DATE": complaints["FAIL_DATE"],
    "CRASH": complaints["CRASH"],
    "FIRE": complaints["FIRE"],
    "INJURED": complaints["INJURED"],
    "DEATHS": complaints["DEATHS"],
}).reset_index(drop=True)

recalls_final = pd.DataFrame({
    "RECORD_ID": recalls["RECORD_ID"].astype("int64"),
    "CAMPNO": recalls["CAMPNO"],
    "MFR_NAME_RAW": recalls["MFGNAME"].str.upper(),
    "MFR_NAME_CLEAN": recalls["MFR_NAME_CLEAN"],
    "VEHICLE_MAKE": recalls["VEHICLE_MAKE"],
    "MODEL_NAME": recalls["MODEL_NAME"],
    "MODEL_NORM": recalls["MODEL_NORM"],
    "MODEL_YEAR": recalls["MODEL_YEAR"],
    "COMPONENT_RAW": recalls["COMPONENT_RAW"],
    "COMPONENT_CLEAN": recalls["COMPONENT_CLEAN"],
    "RECALL_DATE": recalls["RECALL_DATE"],
    "UNITS_AFFECTED": recalls["UNITS_AFFECTED"],
    "INFLUENCED_BY": recalls["INFLUENCED_BY"],
    "DO_NOT_DRIVE": recalls["DO_NOT_DRIVE"],
    "PARK_OUTSIDE": recalls["PARK_OUTSIDE"],
}).reset_index(drop=True)

# One text row per campaign (the text repeats on every vehicle row of a campaign).
recall_text = (recalls.drop_duplicates("CAMPNO")
               [["CAMPNO", "DESC_DEFECT", "CONEQUENCE_DEFECT"]]
               .rename(columns={"DESC_DEFECT": "DEFECT_DESCRIPTION",
                                "CONEQUENCE_DEFECT": "CONSEQUENCE_DESCRIPTION"})  # fixes NHTSA's typo
               .reset_index(drop=True))

complaint_text = None
if LOAD_COMPLAINT_TEXT:
    complaint_text = (complaints.drop_duplicates("ODINO")[["ODINO", "CDESCR"]]
                      .rename(columns={"CDESCR": "COMPLAINT_TEXT"})
                      .astype({"ODINO": "int64"}).reset_index(drop=True))

# Free the working copies before the SQL load; only the final tables are needed from here on.
import gc
del complaints, recalls
gc.collect()

complaints_final.info(memory_usage="deep")
recalls_final.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2042264 entries, 0 to 2042263
Data columns (total 16 columns):
 #   Column           Dtype         
---  ------           -----         
 0   CMPLID           int64         
 1   ODINO            int64         
 2   MFR_NAME_RAW     string        
 3   MFR_NAME_CLEAN   object        
 4   VEHICLE_MAKE     string        
 5   MODEL_NAME       string        
 6   MODEL_NORM       string        
 7   MODEL_YEAR       Int64         
 8   COMPONENT_RAW    string        
 9   COMPONENT_CLEAN  object        
 10  COMPLAINT_DATE   datetime64[ns]
 11  FAIL_DATE        datetime64[ns]
 12  CRASH            int8          
 13  FIRE             int8          
 14  INJURED          int32         
 15  DEATHS           int32         
dtypes: Int64(1), datetime64[ns](2), int32(2), int64(2), int8(2), object(2), string(5)
memory usage: 561.1 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59560 entries, 0 to 59559
Data columns (total 15 columns):
 #

## 11. Validation checks

Each check fails loudly instead of printing a number someone has to eyeball.

In [12]:
checks = {
    "complaint rows unique on CMPLID": complaints_final["CMPLID"].is_unique,
    "complaint key unique": not complaints_final.duplicated(cmpl_key).any(),
    "recall key unique": not recalls_final.duplicated(rcl_key).any(),
    "recall RECORD_ID unique": recalls_final["RECORD_ID"].is_unique,
    "no missing complaint dates": complaints_final["COMPLAINT_DATE"].notna().all(),
    "no missing recall dates": recalls_final["RECALL_DATE"].notna().all(),
    "no future dates": (complaints_final["COMPLAINT_DATE"].max() <= TODAY) and (recalls_final["RECALL_DATE"].max() <= TODAY),
    "every OEM mapped": complaints_final["MFR_NAME_CLEAN"].notna().all() and recalls_final["MFR_NAME_CLEAN"].notna().all(),
    "recall components all appear in complaints": set(recalls_final["COMPONENT_CLEAN"].dropna()) <= set(complaints_final["COMPONENT_CLEAN"]),
    "deaths/injuries non-negative": (complaints_final[["INJURED", "DEATHS"]] >= 0).all().all(),
    "one text row per campaign": recall_text["CAMPNO"].is_unique and len(recall_text) == recalls_final["CAMPNO"].nunique(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "Validation failed; see above"

lineage_df = pd.DataFrame(lineage)
lineage_df.to_csv(OUT_DIR / "cleaning_lineage.csv", index=False)
lineage_df

PASS complaint rows unique on CMPLID
PASS complaint key unique
PASS recall key unique
PASS recall RECORD_ID unique
PASS no missing complaint dates
PASS no missing recall dates
PASS no future dates
PASS every OEM mapped
PASS recall components all appear in complaints
PASS deaths/injuries non-negative
PASS one text row per campaign


,table,step,rows_before,rows_after,rows_removed,note
0,complaints,vehicle records only (PROD_TYPE = V),2234970,2162988,71982,
1,recalls,vehicle recalls only (RCLTYPECD = V),326100,285005,41095,
2,complaints,"valid, non-future complaint date",2162988,2162988,0,
3,recalls,"valid, non-future recall date",285005,285005,0,
4,complaints,make maps to a passenger-vehicle OEM,2162988,2108157,54831,
5,recalls,make maps to a passenger-vehicle OEM,285005,149322,135683,
6,complaints,drop blank / free-text component,2108157,2107601,556,
7,complaints,one row per complaint x vehicle x component,2107601,2042264,65337,
8,recalls,one row per campaign x vehicle x component,149322,59560,89762,


## 12. Load into SQL Server

The tables are written to database `NHTSA_v2`, schema `v2`. The database is created on D: the first time
the notebook runs. Tables are replaced on each run, so the notebook is safe to re-run.

Next step: run the scripts in `sql/` in numeric order (or `sql/run_all.ps1`).

In [13]:
if LOAD_TO_SQL:
    from urllib.parse import quote_plus
    import sqlalchemy as sa
    from sqlalchemy import types as T

    def make_engine(database: str, **kw):
        odbc = (f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_SERVER};"
                f"DATABASE={database};Trusted_Connection=yes;")
        return sa.create_engine("mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc), **kw)

    # Create the database on D: the first time. SIMPLE recovery keeps the log small.
    Path(SQL_DATA_DIR).mkdir(parents=True, exist_ok=True)
    master = make_engine("master", isolation_level="AUTOCOMMIT")
    with master.connect() as conn:
        exists = conn.exec_driver_sql(f"SELECT DB_ID('{SQL_DATABASE}')").scalar()
        if exists is None:
            conn.exec_driver_sql(f"""
                CREATE DATABASE [{SQL_DATABASE}]
                ON (NAME = {SQL_DATABASE}, FILENAME = '{SQL_DATA_DIR}\\{SQL_DATABASE}.mdf', SIZE = 2GB, FILEGROWTH = 512MB)
                LOG ON (NAME = {SQL_DATABASE}_log, FILENAME = '{SQL_DATA_DIR}\\{SQL_DATABASE}_log.ldf', SIZE = 512MB, FILEGROWTH = 256MB)""")
            conn.exec_driver_sql(f"ALTER DATABASE [{SQL_DATABASE}] SET RECOVERY SIMPLE")
            print(f"Created database {SQL_DATABASE} in {SQL_DATA_DIR}")
    master.dispose()

    # fast_executemany sends rows in bulk and is ~10x faster, but pyodbc reserves the column's maximum
    # size for every row, which exhausts memory on VARCHAR(MAX) columns. Tables with long text use
    # the plain engine instead.
    engine = make_engine(SQL_DATABASE, fast_executemany=True)
    engine_long_text = make_engine(SQL_DATABASE)

    with engine.begin() as conn:
        conn.exec_driver_sql(f"IF SCHEMA_ID('{SQL_SCHEMA}') IS NULL EXEC('CREATE SCHEMA {SQL_SCHEMA}')")

    def width(s: pd.Series, floor: int = 10) -> int:
        return max(floor, int(s.dropna().astype(str).str.len().max() or 0))

    def is_text(s: pd.Series) -> bool:
        return pd.api.types.is_string_dtype(s) and not pd.api.types.is_datetime64_any_dtype(s)

    def text_types(df: pd.DataFrame) -> dict:
        out = {}
        for col in df.columns:
            if is_text(df[col]):
                w = width(df[col])
                out[col] = T.VARCHAR(w) if w <= 8000 else T.VARCHAR(None)  # None -> VARCHAR(MAX)
        return out

    def load(df: pd.DataFrame, table: str, dtypes: dict, batch: int = 100_000):
        """Load in batches, each its own transaction: memory stays flat, and an interrupted
        load doesn't leave SQL Server rolling back millions of rows."""
        t0 = time.time()
        types = {**text_types(df), **dtypes}
        has_max = any(isinstance(t, T.VARCHAR) and t.length is None for t in types.values())
        eng = engine_long_text if has_max else engine
        text_cols = [c for c in df.columns if is_text(df[c])]
        for start in range(0, len(df), batch):
            chunk = df.iloc[start:start + batch].copy()
            chunk[text_cols] = chunk[text_cols].astype(object).where(chunk[text_cols].notna(), None)
            chunk.to_sql(table, eng, schema=SQL_SCHEMA, index=False, dtype=types,
                         if_exists="replace" if start == 0 else "append",
                         method=None if not has_max else "multi", chunksize=None if not has_max else 50)
        print(f"{SQL_SCHEMA}.{table}: {len(df):,} rows in {time.time() - t0:.0f}s")

    load(complaints_final, "complaints", {
        "CMPLID": T.BIGINT, "ODINO": T.BIGINT, "MODEL_YEAR": T.SMALLINT,
        "COMPLAINT_DATE": T.DATE, "FAIL_DATE": T.DATE, "CRASH": T.SMALLINT, "FIRE": T.SMALLINT,
        "INJURED": T.INTEGER, "DEATHS": T.INTEGER})
    load(recalls_final, "recalls", {
        "RECORD_ID": T.BIGINT, "MODEL_YEAR": T.SMALLINT, "RECALL_DATE": T.DATE,
        "UNITS_AFFECTED": T.BIGINT, "DO_NOT_DRIVE": T.SMALLINT, "PARK_OUTSIDE": T.SMALLINT})
    load(recall_text, "recall_text", {})
    if complaint_text is not None:
        load(complaint_text, "complaint_text", {"ODINO": T.BIGINT})

    with engine.connect() as conn:
        for t in ["complaints", "recalls", "recall_text"]:
            n = conn.exec_driver_sql(f"SELECT COUNT(*) FROM {SQL_SCHEMA}.{t}").scalar()
            print(f"Read back {SQL_SCHEMA}.{t}: {n:,} rows")

v2.complaints: 2,042,264 rows in 219s
v2.recalls: 59,560 rows in 10s
v2.recall_text: 11,059 rows in 2s
Read back v2.complaints: 2,042,264 rows
Read back v2.recalls: 59,560 rows
Read back v2.recall_text: 11,059 rows
